In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import (
    shapiro, ttest_ind, mannwhitneyu,
    chi2_contingency, levene
)
from scipy.stats import norm
from scipy.stats import ncx2

warnings.filterwarnings('ignore')

In [2]:
filename_patient = Path('..') / 'data' / 'patient.csv'
filename_station = Path('..') / 'data' / 'station.csv'
filename_observation = Path('..') / 'data' / 'observation.csv'

observation_df = pd.read_csv(filename_observation, sep='\t')
patient_df = pd.read_csv(filename_patient, sep='\t')
station_df = pd.read_csv(filename_station, sep='\t')

print("Data loaded successfully")
print(f"\nNumber of records:")
print(f"  Observation: {len(observation_df):,}")
print(f"  Patient: {len(patient_df):,}")
print(f"  Station: {len(station_df):,}")

Data loaded successfully

Number of records:
  Observation: 12,133
  Patient: 2,154
  Station: 746


In [3]:
patient_df['address'] = patient_df['address'].str.replace('\n', ', ').str.replace('\r', '')
patient_df.head()

,blood_group,user_id,company,registration,address,job,ssn,username,residence,name,current_location,mail,station_ID
0,B-,1384,Wulf Heinz AG,2024/06/13,"Zänkerweg 6-2, 75317 Pößneck",NaN,079-86-6480,ilias32,NaN,Herr Pirmin Stadelmann B.A.,"(Decimal('-9.5081185'), Decimal('-108.465353'))",bruno18@gmx.de,289
1,O+,1398,Borges Moreira Ltda.,"10/11/2024, 00:00:00","Fazenda Brenda Vieira, 51, Piratininga, 99724-...",NaN,84926073196,pedro-miguelvargas,NaN,Sr. Matheus Cirino,"(Decimal('38.5020005'), Decimal('151.185055'))",ana-beatrizpacheco@hotmail.com,326
2,B-,163,高橋建設有限会社,"10/19/2022, 00:00:00",島根県豊島区上高野24丁目16番6号,NaN,127-45-0018,skobayashi,NaN,中村 真綾,"(Decimal('87.3250985'), Decimal('-83.533367'))",vmaeda@yahoo.com,594
3,B+,112,Yang-Gray,2019/01/19,"93041 Wright Turnpike, Lake Loritown, RI 96307",NaN,765-77-3956,kochmario,NaN,Collin Wright,"(Decimal('-45.2256685'), Decimal('147.973684'))",reaton@yahoo.com,738
4,A+,92,Johnson Ltd,2019/11/10,"0089 William Run, West Adam, TX 90462",NaN,457-20-6978,paul10,NaN,Kristina Murray,"(Decimal('7.750758'), Decimal('71.027557'))",cameron61@hotmail.com,628


In [4]:
patient_df['registration'] = (
    patient_df['registration']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.strip()
)

patient_df['registration'] = pd.to_datetime(
    patient_df['registration'],
    errors='coerce',
    format='mixed'
)
patient_df.head()

,blood_group,user_id,company,registration,address,job,ssn,username,residence,name,current_location,mail,station_ID
0,B-,1384,Wulf Heinz AG,2024-06-13,"Zänkerweg 6-2, 75317 Pößneck",NaN,079-86-6480,ilias32,NaN,Herr Pirmin Stadelmann B.A.,"(Decimal('-9.5081185'), Decimal('-108.465353'))",bruno18@gmx.de,289
1,O+,1398,Borges Moreira Ltda.,2024-10-11,"Fazenda Brenda Vieira, 51, Piratininga, 99724-...",NaN,84926073196,pedro-miguelvargas,NaN,Sr. Matheus Cirino,"(Decimal('38.5020005'), Decimal('151.185055'))",ana-beatrizpacheco@hotmail.com,326
2,B-,163,高橋建設有限会社,2022-10-19,島根県豊島区上高野24丁目16番6号,NaN,127-45-0018,skobayashi,NaN,中村 真綾,"(Decimal('87.3250985'), Decimal('-83.533367'))",vmaeda@yahoo.com,594
3,B+,112,Yang-Gray,2019-01-19,"93041 Wright Turnpike, Lake Loritown, RI 96307",NaN,765-77-3956,kochmario,NaN,Collin Wright,"(Decimal('-45.2256685'), Decimal('147.973684'))",reaton@yahoo.com,738
4,A+,92,Johnson Ltd,2019-11-10,"0089 William Run, West Adam, TX 90462",NaN,457-20-6978,paul10,NaN,Kristina Murray,"(Decimal('7.750758'), Decimal('71.027557'))",cameron61@hotmail.com,628


In [5]:
def analyze_missing_values(df, dataset_name):
    missing = pd.DataFrame({
        'Column': df.columns,
        'Missing_Count': df.isnull().sum().values,
        'Missing_%': (df.isnull().sum().values / len(df) * 100).round(2)
    })
    missing = missing[missing['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

    print(f"\n{dataset_name}:")
    if len(missing) == 0:
        print("No missing values")
    else:
        print(f"Total number of columns with missing values: {len(missing)}")
        display(missing)

    return missing

missing_obs = analyze_missing_values(observation_df, "Observation")
missing_pat = analyze_missing_values(patient_df, "Patient")
missing_sta = analyze_missing_values(station_df, "Station")



Observation:
No missing values

Patient:
Total number of columns with missing values: 4


,Column,Missing_Count,Missing_%
8,residence,2154,100.00
5,job,1508,70.01
4,address,323,15.00
10,current_location,108,5.01



Station:
No missing values


In [6]:
patient_df = patient_df.drop(columns=['residence'])
patient_df = patient_df.drop(columns=['job'])

In [7]:
station_df[['continent', 'city']] = station_df['location'].str.rsplit('/', n=1, expand=True)
station_df = station_df.drop(columns=['location'])
station_df.head()

,station,longitude,QoS,latitude,revision,continent,city
0,Kenda,86.51499,good,23.19590,17 Jun 2018,Asia,Kolkata
1,Canton,-83.48216,good,42.30865,2022/10/10,America,Detroit
2,Zaysan,84.87144,good,47.46657,2016/11/06,Asia,Almaty
3,Shushary,30.38167,good,59.80917,2022/09/10,Europe,Moscow
4,Cheraga,2.95924,good,36.76775,07 Jan 2024,Africa,Algiers


In [8]:
station_df['revision'] = (
    station_df['revision']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.strip()
)

station_df['revision'] = pd.to_datetime(
    station_df['revision'],
    errors='coerce',
    format='mixed'
)
station_df.head()

,station,longitude,QoS,latitude,revision,continent,city
0,Kenda,86.51499,good,23.19590,2018-06-17,Asia,Kolkata
1,Canton,-83.48216,good,42.30865,2022-10-10,America,Detroit
2,Zaysan,84.87144,good,47.46657,2016-11-06,Asia,Almaty
3,Shushary,30.38167,good,59.80917,2022-09-10,Europe,Moscow
4,Cheraga,2.95924,good,36.76775,2024-01-07,Africa,Algiers


In [ ]:
from sklearn.model_selection import train_test_split

X = observation_df.drop('oximetry', axis=1)
y = observation_df['oximetry']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PowerTransformer, QuantileTransformer
import pandas as pd

scaler_std = StandardScaler()
X_train_std = scaler_std.fit_transform(X_train)


scaler_mm = MinMaxScaler()
X_train_mm = scaler_mm.fit_transform(X_train)

pt = PowerTransformer()
X_train_pt = pt.fit_transform(X_train)

qt = QuantileTransformer(output_distribution='uniform')
X_train_qt = qt.fit_transform(X_train)

In [ ]:
import numpy as np

# Сравним распределение до и после масштабирования
print("Оригинальные данные:")
display(pd.DataFrame(X_train).describe().T[['mean', 'std', 'min', 'max']])

print("\nПосле StandardScaler:")
display(pd.DataFrame(X_train_std).describe().T[['mean', 'std', 'min', 'max']])

print("\nПосле MinMaxScaler:")
display(pd.DataFrame(X_train_mm).describe().T[['mean', 'std', 'min', 'max']])

# Проверим PowerTransformer и QuantileTransformer
print("\nПосле PowerTransformer:")
display(pd.DataFrame(X_train_pt).describe().T[['mean', 'std', 'min', 'max']])

print("\nПосле QuantileTransformer:")
display(pd.DataFrame(X_train_qt).describe().T[['mean', 'std', 'min', 'max']])

### 2.1 Realizácia predspracovania dát

V tejto fáze boli dáta rozdelené na tréningovú (80 %) a testovaciu (20 %) množinu pomocou `train_test_split`.  
Cieľová premenná je `oximetry`, ostatné atribúty sú numerické (`float64`).

Pre spracovanie údajov boli aplikované nasledujúce štyri techniky:

- **StandardScaler** – normalizuje hodnoty na priemer 0 a smerodajnú odchýlku 1, čím odstraňuje rozdiel v jednotkách merania.  
- **MinMaxScaler** – prevádza hodnoty do intervalu [0,1], čo je vhodné napríklad pre neurónové siete.  
- **PowerTransformer (Yeo–Johnson)** – upravuje distribúciu atribútov tak, aby sa priblížila k normálnemu rozdeleniu.  
- **QuantileTransformer (uniform)** – transformuje dáta na rovnomerné rozdelenie, čo redukuje vplyv extrémnych hodnôt (outlierov).

Porovnaním popisných štatistík pred a po transformáciách vidno, že škálovanie prebehlo korektne:  
pri **StandardScaler** sa stredné hodnoty približujú k nule, pri **MinMaxScaler** a **QuantileTransformer** sa hodnoty nachádzajú v rozsahu [0,1].  
Tieto transformácie zlepšia stabilitu a výkon modelov strojového učenia a budú integrované do `Pipeline` vo fáze 2.3.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.ensemble import RandomForestRegressor

# 1️⃣ Корреляция с целевой переменной
corr = observation_df.corr(numeric_only=True)['oximetry'].sort_values(ascending=False)
print("Корреляция с oximetry:")
display(corr)
print(corr)

# 2️⃣ SelectKBest (f_regression)
X = observation_df.drop('oximetry', axis=1)
y = observation_df['oximetry']

selector = SelectKBest(score_func=f_regression, k='all')
selector.fit(X, y)
kbest_scores = pd.Series(selector.scores_, index=X.columns).sort_values(ascending=False)
print("\nSelectKBest результаты:")
display(kbest_scores)
print(kbest_scores)

# 3️⃣ RandomForestRegressor feature importance
rf = RandomForestRegressor(random_state=42)
rf.fit(X, y)
rf_importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nRandomForest feature importance:")
display(rf_importance)
print(rf_importance)


### 2.2 Výber atribútov pre strojové učenie

Pre určenie informatívnych atribútov k predikovanej premennej `oximetry` boli použité tri techniky:

1. **Korelačná analýza** – zisťuje lineárnu závislosť medzi atribútmi a cieľovou premennou.  
   Najvyššiu koreláciu s `oximetry` majú: `PVI (0.65)`, `PRV (0.37)`, `HR (0.27)` a `CO (0.11)`.  
   Atribúty ako `latitude` a `BP` nemajú prakticky žiadny vplyv.

2. **SelectKBest (f_regression)** – vyhodnocuje štatistickú významnosť atribútov.  
   Najvýznamnejšie sú: `PVI`, `PRV`, `HR`, `EtCO₂`, `CO` a `SpO₂`.

3. **RandomForestRegressor** – určuje dôležitosť atribútov podľa príspevku k presnosti modelu.  
   Najvyššiu dôležitosť majú: `PVI`, `PRV`, `Motion/Activity index`, `EtCO₂` a `CO`.

Porovnaním výsledkov je možné určiť, že najdôležitejšie atribúty pre predikciu `oximetry` sú:
**PVI, PRV, HR, CO a EtCO₂**.  
Tieto atribúty budú použité v ďalšej fáze (3. modelovanie), zatiaľ čo málo informatívne atribúty (`BP`, `latitude`, `longitude`) budú vylúčené.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.model_selection import train_test_split

# Разделение данных
X = observation_df.drop('oximetry', axis=1)
y = observation_df['oximetry']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Создаём pipeline
preprocessing_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('transformer', PowerTransformer())
])

# Обучаем pipeline на train
X_train_processed = preprocessing_pipeline.fit_transform(X_train)

# Применяем тот же pipeline к test
X_test_processed = preprocessing_pipeline.transform(X_test)

print("Train shape:", X_train_processed.shape)
print("Test shape:", X_test_processed.shape)

### 2.3 Replikovateľnosť predspracovania údajov

Na zabezpečenie opakovateľnosti predspracovania údajov bol vytvorený `sklearn.Pipeline`,  
ktorý spája jednotlivé kroky (škálovanie a transformácie) do jednej sekvencie.  
Pipeline bol natrénovaný na tréningovej množine (`fit`) a následne aplikovaný na testovaciu množinu (`transform`).

Použité techniky:
- **StandardScaler** – normalizácia atribútov,
- **PowerTransformer (Yeo–Johnson)** – stabilizácia rozdelenia dát.

Vďaka použitiu `Pipeline` je možné rovnaké spracovanie použiť opakovane na akúkoľvek množinu dát  
bez potreby meniť kód, čím sa zabezpečuje reprodukovateľnosť a konzistentnosť celého procesu.